In [ ]:
import sys
sys.modules['tensorflow'] = None

import os
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("TRANSFORMERS_NO_FLAX", "1")
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModel

In [ ]:
TSV_PATH = "../data/unique_TCRs.tsv"
# Process both alpha and beta chains; embeddings are concatenated → 768*2 = 1536 dims
SEQ_COLS = ["TCR_alpha_CDR3", "TCR_beta_CDR3"]
MODEL_ID = "wukevin/tcr-bert"
BATCH_SIZE = 64  # Reduce if out of memory
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_BASENAME = "TCR_tcrbert_embeddings"

In [ ]:
VALID_AA = set(list("ACDEFGHIKLMNPQRSTVWY"))

def clean_seq(seq: str) -> str:
    """Remove invalid characters and keep only standard amino acids."""
    s = (str(seq) or "").strip().replace(" ", "").upper()
    s = "".join(c for c in s if c in VALID_AA)
    return s

def to_spaced(seq: str) -> str:
    """Convert cleaned sequence to space-separated format for tokenizer."""
    s = clean_seq(seq)
    return " ".join(list(s))

def chunks(lst, n):
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def read_tsv_robust(path: str) -> pd.DataFrame:
    """Robustly read TSV file with automatic encoding detection."""
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1", "gb18030"]
    last_err = None
    
    for enc in encodings:
        try:
            df = pd.read_csv(path, sep='\t', encoding=enc)
            # Clean column names (remove BOM, non-breaking spaces)
            df.columns = [
                str(c).replace("\ufeff", "").replace("\xa0", " ").strip()
                for c in df.columns
            ]
            return df
        except UnicodeDecodeError as e:
            last_err = e
            continue
        except Exception as e:
            last_err = e
            continue

    try:
        df = pd.read_csv(path, sep='\t', encoding='utf-8', errors='replace')
        df.columns = [
            str(c).replace("\ufeff", "").replace("\xa0", " ").strip()
            for c in df.columns
        ]
        return df
    except Exception as e:
        raise e from last_err
    
print(f"Reading TSV file: {TSV_PATH}")
df = read_tsv_robust(TSV_PATH)
print(f"Loaded {len(df)} rows with columns: {list(df.columns)}")

In [ ]:
lowered_map = {c.lower(): c for c in df.columns}
resolved_cols = []
missing = []

for col in SEQ_COLS:
    key = str(col).lower()
    if key in lowered_map:
        resolved_cols.append(lowered_map[key])
    else:
        missing.append(col)

if missing:
    raise ValueError(
        f"Columns not found in data: {missing}\nAvailable columns: {list(df.columns)}"
    )

print(f"Processing columns: {resolved_cols}")

In [ ]:
print(f"Loading model: {MODEL_ID}")
tok = AutoTokenizer.from_pretrained(MODEL_ID)
bert = AutoModel.from_pretrained(MODEL_ID).to(DEVICE)
bert.eval()

# Verify embedding dimension
with torch.no_grad():
    test_input = tok("A C D", return_tensors="pt").to(DEVICE)
    test_output = bert(**test_input).last_hidden_state
    EMBEDDING_DIM = test_output.shape[-1]
    print(f"Model embedding dimension: {EMBEDDING_DIM}")

In [ ]:
def compute_embeddings(spaced_tokens_list):
    """Compute embeddings for a list of space-separated sequences."""
    emb_list = []
    with torch.no_grad():
        for batch in tqdm(list(chunks(spaced_tokens_list, BATCH_SIZE)), desc="Computing embeddings"):
            toks = tok(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
            ).to(DEVICE)
            out = bert(**toks).last_hidden_state  # [B, L, D]
            mask = toks["attention_mask"].unsqueeze(-1)  # [B, L, 1]
            # Mean pooling over sequence length
            vec = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)  # [B, D]
            emb_list.append(vec.cpu())
    return torch.cat(emb_list, dim=0).numpy()

In [ ]:
N = len(df)
all_emb_blocks = []  # List of [N, EMBEDDING_DIM] arrays
all_emb_colnames = []

for col in resolved_cols:
    print(f"\nProcessing column: {col}")
    
    # Clean sequences
    seq_clean = df[col].map(clean_seq)
    mask_nonempty = seq_clean.str.len() > 0
    
    # Statistics
    original_count = df[col].notna().sum()
    valid_count = mask_nonempty.sum()
    invalid_count = original_count - valid_count
    
    if invalid_count > 0:
        print(f"  Warning: {invalid_count}/{original_count} sequences became empty after cleaning")
    
    idxs = np.where(mask_nonempty.values)[0]
    inputs_spaced = seq_clean[mask_nonempty].map(to_spaced).tolist()
    
    if len(inputs_spaced) == 0:
        # All sequences are empty, fill with NaN
        emb_col = np.full((N, EMBEDDING_DIM), np.nan, dtype=float)
        all_emb_blocks.append(emb_col)
        all_emb_colnames.extend([f"{col}_dim_{i:03d}" for i in range(EMBEDDING_DIM)])
        print(f"  Warning: No valid sequences in column {col}, filling with NaN")
        continue
    
    # Compute embeddings for valid sequences
    print(f"  Computing embeddings for {len(inputs_spaced)} valid sequences")
    emb_valid = compute_embeddings(inputs_spaced)  # [M, EMBEDDING_DIM]
    
    # Create full column with NaN for invalid positions
    emb_col = np.full((N, EMBEDDING_DIM), np.nan, dtype=float)
    emb_col[idxs, :] = emb_valid
    
    all_emb_blocks.append(emb_col)
    all_emb_colnames.extend([f"{col}_dim_{i:03d}" for i in range(EMBEDDING_DIM)])

# 4) Concatenate all embeddings
if all_emb_blocks:
    emb_all = np.concatenate(all_emb_blocks, axis=1)
else:
    emb_all = np.empty((N, 0), dtype=float)

print(f"\nFinal embedding shape: {emb_all.shape}")

# 5) Save outputs

from pathlib import Path

OUT_DIR = Path("../data")
OUT_DIR.mkdir(exist_ok=True)

OUT_NPY = str(OUT_DIR / f"{OUT_BASENAME}.npy")
np.save(OUT_NPY, emb_all)
print(f"Saved: {OUT_NPY}")